# Weather Across the Season at Globe Life Field

**Goal:** Visualize how temperature, pressure, humidity, and wind speed vary by month at Globe Life Field (Texas Rangers) to understand seasonal weather patterns and their potential effects on run scoring.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load league-wide game data and filter to TEX home games
data = pd.read_csv('../../data/league_weather_2021_2025.csv')
tex = data[data['home_team'] == 'TEX'].copy()

# Parse game_date to extract month
tex['game_date'] = pd.to_datetime(tex['game_date'])
tex['month'] = tex['game_date'].dt.month
month_map = {3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
             7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct'}
tex['month_name'] = tex['month'].map(month_map)

months = sorted(tex['month'].unique())
month_names = [month_map.get(m, str(m)) for m in months]

print(f"Total TEX home games: {len(tex)}")
print(f"\nGames per month:")
print(tex.groupby('month_name').size().reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Weather Distributions by Month
# ============================================================

weather_vars = {
    'temp_f':   {'label': 'Temperature (\u00b0F)', 'color': '#d62728'},
    'rhum':     {'label': 'Humidity (%)',      'color': '#1f77b4'},
    'wspd_mph': {'label': 'Wind Speed (mph)',  'color': '#2ca02c'},
    'pres':     {'label': 'Pressure (hPa)',    'color': '#9467bd'},
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (col, info) in zip(axes.flat, weather_vars.items()):
    box_data = [tex.loc[tex['month'] == m, col].dropna().values for m in months]
    bp = ax.boxplot(box_data, patch_artist=True, labels=month_names,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch in bp['boxes']:
        patch.set_facecolor(info['color'])
        patch.set_alpha(0.6)
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel(info['label'], fontsize=11)
    ax.set_title(info['label'], fontsize=13)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Weather Distributions by Month \u2014 Globe Life Field', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Scatter Plots: Temperature vs. Pressure & Temperature vs. Humidity
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Temperature vs. Pressure
ax = axes[0]
scatter = ax.scatter(tex['temp_f'], tex['pres'], c=tex['month'], cmap='coolwarm',
                     alpha=0.6, edgecolors='black', linewidth=0.3, s=40)
z = np.polyfit(tex['temp_f'].dropna(), tex.loc[tex['temp_f'].notna(), 'pres'], 1)
p = np.poly1d(z)
x_line = np.linspace(tex['temp_f'].min(), tex['temp_f'].max(), 100)
ax.plot(x_line, p(x_line), color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Temperature (\u00b0F)', fontsize=11)
ax.set_ylabel('Pressure (hPa)', fontsize=11)
ax.set_title('Temperature vs. Pressure', fontsize=13)
ax.yaxis.grid(True, alpha=0.3)
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

# Temperature vs. Humidity
ax = axes[1]
scatter = ax.scatter(tex['temp_f'], tex['rhum'], c=tex['month'], cmap='coolwarm',
                     alpha=0.6, edgecolors='black', linewidth=0.3, s=40)
z = np.polyfit(tex['temp_f'].dropna(), tex.loc[tex['temp_f'].notna(), 'rhum'], 1)
p = np.poly1d(z)
ax.plot(x_line, p(x_line), color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('Temperature (\u00b0F)', fontsize=11)
ax.set_ylabel('Humidity (%)', fontsize=11)
ax.set_title('Temperature vs. Humidity', fontsize=13)
ax.yaxis.grid(True, alpha=0.3)
ax.xaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)

cbar = fig.colorbar(scatter, ax=axes, label='Month', ticks=months)
cbar.ax.set_yticklabels(month_names)

fig.suptitle('Weather Relationships \u2014 Globe Life Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Day vs. Night classification
# ============================================================

# Day games start before 5 PM, night games at 5 PM or later
tex['time_of_day'] = tex['start_hour'].apply(lambda h: 'Day' if h < 17 else 'Night')

print("Game counts by month and time of day:")
print(tex.groupby(['month_name', 'time_of_day']).size().unstack(fill_value=0)
      .reindex([month_map[m] for m in months]))

In [ ]:
# ============================================================
# Average Temperature: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = tex[tex['time_of_day'] == label]
    grouped = subset.groupby('month')['temp_f']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.3,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Temperature (\u00b0F)', fontsize=12)
ax.set_title('Average Temperature: Day vs. Night Games by Month \u2014 Globe Life Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Humidity: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = tex[tex['time_of_day'] == label]
    grouped = subset.groupby('month')['rhum']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.3,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Humidity (%)', fontsize=12)
ax.set_title('Average Humidity: Day vs. Night Games by Month \u2014 Globe Life Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Pressure: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

all_means = []
all_sems = []
for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = tex[tex['time_of_day'] == label]
    grouped = subset.groupby('month')['pres']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    all_means.append(means)
    all_sems.append(sems)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

# Zoom y-axis to better show differences
y_min = min(m.min() for m in all_means) - 2
y_max = max(m.max() + s.max() for m, s in zip(all_means, all_sems)) + 1.5
ax.set_ylim(y_min, y_max)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Pressure (hPa)', fontsize=12)
ax.set_title('Average Pressure: Day vs. Night Games by Month \u2014 Globe Life Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Wind Speed: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = tex[tex['time_of_day'] == label]
    grouped = subset.groupby('month')['wspd_mph']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Wind Speed (mph)', fontsize=12)
ax.set_title('Average Wind Speed: Day vs. Night Games by Month \u2014 Globe Life Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Typical Day Game Wind Direction by Month (Polar Rose)
# ============================================================

def deg_to_compass(deg):
    dirs = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
    idx = int((deg + 22.5) % 360 / 45)
    return dirs[idx]

compass_order = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']
compass_angles = {d: np.radians(i * 45) for i, d in enumerate(compass_order)}

day_games = tex[tex['time_of_day'] == 'Day'].copy()
day_games['compass'] = day_games['wdir'].apply(deg_to_compass)

active_months = sorted(day_games['month'].unique())
n_months = len(active_months)
cols = min(4, n_months)
rows = int(np.ceil(n_months / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows),
                         subplot_kw=dict(projection='polar'))
if n_months == 1:
    axes = np.array([axes])
axes = axes.flat

for idx, m in enumerate(active_months):
    ax = axes[idx]
    subset = day_games[day_games['month'] == m]
    counts = subset['compass'].value_counts().reindex(compass_order, fill_value=0)
    angles = [compass_angles[d] for d in compass_order]
    bars = ax.bar(angles, counts, width=np.radians(40), alpha=0.7,
                  color='#2ca02c', edgecolor='black', linewidth=0.5)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_thetagrids([i * 45 for i in range(8)], compass_order, fontsize=9)
    ax.set_title(f'{month_map[m]}  (n={len(subset)})', fontsize=11, pad=15)
    ax.set_yticklabels([])

for idx in range(n_months, rows * cols):
    axes[idx].set_visible(False)

fig.suptitle('Day Game Wind Direction by Month \u2014 Globe Life Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Typical Night Game Wind Direction by Month (Polar Rose)
# ============================================================

night_games = tex[tex['time_of_day'] == 'Night'].copy()
night_games['compass'] = night_games['wdir'].apply(deg_to_compass)

active_months_n = sorted(night_games['month'].unique())
n_months_n = len(active_months_n)
cols = min(4, n_months_n)
rows = int(np.ceil(n_months_n / cols))

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows),
                         subplot_kw=dict(projection='polar'))
if n_months_n == 1:
    axes = np.array([axes])
axes = axes.flat

for idx, m in enumerate(active_months_n):
    ax = axes[idx]
    subset = night_games[night_games['month'] == m]
    counts = subset['compass'].value_counts().reindex(compass_order, fill_value=0)
    angles = [compass_angles[d] for d in compass_order]
    bars = ax.bar(angles, counts, width=np.radians(40), alpha=0.7,
                  color='#003366', edgecolor='black', linewidth=0.5)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_thetagrids([i * 45 for i in range(8)], compass_order, fontsize=9)
    ax.set_title(f'{month_map[m]}  (n={len(subset)})', fontsize=11, pad=15)
    ax.set_yticklabels([])

for idx in range(n_months_n, rows * cols):
    axes[idx].set_visible(False)

fig.suptitle('Night Game Wind Direction by Month \u2014 Globe Life Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Average Total Runs: Day vs. Night Games by Month
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(months))
width = 0.35

for i, (label, color) in enumerate([('Day', '#ff9900'), ('Night', '#003366')]):
    subset = tex[tex['time_of_day'] == label]
    grouped = subset.groupby('month')['total_runs']
    means = grouped.mean().reindex(months)
    sems = grouped.sem().reindex(months)
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=sems, capsize=4,
           label=label, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
    for xi, m in zip(x, means):
        if pd.notna(m):
            ax.text(xi + offset, m + sems.reindex(months).iloc[xi] + 0.1,
                    f'{m:.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(month_names, fontsize=11)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Total Runs', fontsize=12)
ax.set_title('Average Total Runs: Day vs. Night Games by Month \u2014 Globe Life Field', fontsize=13)
ax.legend(fontsize=11)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Correlations with Pressure: Runs, Home Runs, Strikeouts
# ============================================================

corr_vars = {
    'total_runs':     {'label': 'Total Runs',     'color': '#d62728'},
    'home_runs_hit':  {'label': 'Home Runs',       'color': '#ff7f0e'},
    'strikeouts':     {'label': 'Strikeouts',      'color': '#1f77b4'},
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (col, info) in zip(axes, corr_vars.items()):
    valid = tex.dropna(subset=['pres', col])
    r = valid['pres'].corr(valid[col])
    ax.scatter(valid['pres'], valid[col], alpha=0.5, s=30,
               color=info['color'], edgecolors='black', linewidth=0.3)
    # Trend line
    z = np.polyfit(valid['pres'], valid[col], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid['pres'].min(), valid['pres'].max(), 100)
    ax.plot(x_line, p(x_line), color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Pressure (hPa)', fontsize=11)
    ax.set_ylabel(info['label'], fontsize=11)
    ax.set_title(f'{info["label"]} vs. Pressure (r = {r:.3f})', fontsize=13)
    ax.xaxis.grid(True, alpha=0.3)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)

fig.suptitle('Correlations with Pressure \u2014 Globe Life Field', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print correlation summary
print("Pearson correlations with pressure:")
for col, info in corr_vars.items():
    valid = tex.dropna(subset=['pres', col])
    r = valid['pres'].corr(valid[col])
    print(f"  {info['label']:15s}: r = {r:+.4f}  (n = {len(valid)})")

In [ ]:
# ============================================================
# Summary Statistics Table
# ============================================================

summary = tex.groupby('month').agg(
    games=('temp_f', 'size'),
    temp_mean=('temp_f', 'mean'),
    temp_std=('temp_f', 'std'),
    rhum_mean=('rhum', 'mean'),
    rhum_std=('rhum', 'std'),
    wspd_mean=('wspd_mph', 'mean'),
    wspd_std=('wspd_mph', 'std'),
    pres_mean=('pres', 'mean'),
    pres_std=('pres', 'std'),
).round(2)

summary.index = [month_map.get(m, str(m)) for m in summary.index]
summary.index.name = 'Month'
summary.columns = ['Games', 'Temp Mean (\u00b0F)', 'Temp Std',
                    'Humidity Mean (%)', 'Humidity Std',
                    'Wind Mean (mph)', 'Wind Std',
                    'Pressure Mean (hPa)', 'Pressure Std']
summary